# ch13 连接与合并：`merge` / `join` / `concat` / `merge_asof`

> 方向：数据预处理 ｜ 竞赛对应：**数据准备及处理**（宽表拼装）+ **数据探索**
> —— 竞赛给的数据永远是"主表 + 若干维表"，
> 能不能把台账、汇总、限值几张表干净地拼到主表上，几乎决定了后面所有步骤。

本章要解决的四个静默错误：

1. **键"看着一样"但匹配 0 行**——多一个空格、大小写不同、类型不同
2. **一对多被当成一对一**——行数悄悄膨胀，均值/计数全错
3. **`concat` 默认保留原索引**——索引重复后 `loc` 一次返回多行
4. **`merge_asof` 的 `direction`** ——`backward` / `forward` 语义相反，选错就是"用了未来数据"

## 一、本节考点

| 竞赛评分点 | 分值含义 | 本节覆盖的操作 |
|---|---|---|
| 数据准备及处理 | 占技能操作 10% | 主表 + 维表拼装成宽表 |
| 数据清洗 | 每项 2 分 | 连接后的缺失识别与填充决策 |
| 数据探索 | 每项 2 分 | 用非空率反查连接是否命中 |
| 结果交付 | 提交格式错 = 白做 | 索引、列序、列名后缀的整理 |

**为什么单独成章**：连接是**唯一一类"错了也能出结果"的操作**。
筛选错会少行、聚合错会少组，这些还能靠行数对账发现；
但连接错只会让某个字段**整列变成 `NaN`**，或者让行数**多出几十行**——
而"多出几十行"在一张 270 行的表里非常容易被忽略，
直到你发现某组的均值怎么算都对不上。

## 二、API 速查表

### 2.1 三个入口的分工

| 入口 | 连接键 | 适用场景 |
|---|---|---|
| `pd.merge(left, right, on=...)` | **列** | 最通用，键是普通列 |
| `left.merge(right, on=...)` | **列** | 同上，链式写法 |
| `left.join(right, on=..., how=...)` | **索引** 或 `on=` 指定的列 | 右侧已 `set_index` |
| `pd.concat([a, b], axis=0)` | **无键，按位置** | 纵向堆叠（追加行） |
| `pd.concat([a, b], axis=1)` | **按索引标签** | 横向拼接（加列） |
| `pd.merge_asof(left, right, on=...)` | **最近键** | 时序最近匹配（生效时间、报价、限值） |

### 2.2 `how` 的四种语义（本数据实测）

| `how` | 保留谁 | 本数据形状 | 语义 |
|---|---|---|---|
| `"inner"` | 两边都有的键 | `(247, 17)` | 只留匹配上的 |
| `"left"` | 左表全部 | `(270, 17)` | **最常用**，右表填 `NaN` |
| `"right"` | 右表全部 | `(247, 17)` | 等价于左右互换的 `left` |
| `"outer"` | 两边并集 | `(270, 17)` | 全保留（此处右表无独有键，所以等于 `left`） |

### 2.3 关键参数

| 参数 | 取值 | 作用 |
|---|---|---|
| `on` | 列名 / 列表 | 连接键（两表同名） |
| `left_on` / `right_on` | 列名 | 两表键名不同时 |
| `left_index` / `right_index` | `bool` | 用索引当键 |
| `suffixes` | `("_左", "_右")` | **非键**同名列的后缀 |
| `validate` | `"one_to_one"` / `"one_to_many"` / `"many_to_one"` / `"many_to_many"` | **连接前强制校验基数**，不合规直接报错 |
| `indicator` | `True` / 字符串 | 加一列标注每行来自哪边 |
| `sort` | `bool` | 按键排序（默认 `False`） |

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.simplefilter("ignore")
pd.set_option("display.width", 170)
pd.set_option("display.max_columns", 40)

DATA = Path("data")

# 主表：缺陷记录（270 行，18 个台区，每台区多行）
dev = pd.read_csv(DATA / "device_defects.csv")
# 维表：设备台账（16 行，故意少 2 个台区）
info = pd.read_csv(DATA / "device_info.csv")
# 汇总表：台区日均负荷（18 行，与主表台区完全对齐）
summary = pd.read_csv(DATA / "station_summary.csv")

KEY = "台区编号"

# 维表换个中文列名，练「不同列名」的连接
info_cn = info.rename(columns={"容量kVA": "容量"})

# 容量限值调整表：每半月调一次，给 merge_asof 用
limit = pd.DataFrame(
    {
        "生效时间": pd.to_datetime(
            ["2026-01-01", "2026-01-16", "2026-02-01", "2026-02-16", "2026-03-01", "2026-03-16"]
        ),
        "限值": [1000, 1050, 1100, 1120, 1150, 1180],
    }
).sort_values("生效时间").reset_index(drop=True)

# 负荷曲线：取单台区、按时间升序（merge_asof 要求左表键有序）
curve = pd.read_csv(DATA / "load_curve.csv", parse_dates=["时间戳"])
hourly = (
    curve[curve[KEY] == "STATION_A_01"]
    .sort_values("时间戳")
    .reset_index(drop=True)
)

print("pandas", pd.__version__)
print("dev", dev.shape, "| info", info.shape, "| summary", summary.shape, "| hourly", hourly.shape)

In [ ]:
# 脚手架：写在 @@todo 之外，练习版原样保留
def merge_report(result, base_rows):
    """算一条校验公式：结果行数相对左表的膨胀倍数。"""
    return round(len(result) / base_rows, 4)


def match_rate(result, col):
    """右表列的非空率——判断连接命中情况最快的方式。"""
    return round(float(result[col].notna().mean()), 4)


def probe(fn, *args, **kwargs):
    """把「调用可能抛异常」写成返回值，避免在挖空块里写 try/except。

    返回 ("ok", 结果) 或 ("err", 异常类型名)。
    """
    try:
        return "ok", fn(*args, **kwargs)
    except Exception as exc:
        return "err", type(exc).__name__


def audit_merge(left, right, key, probe_col):
    """对一次 many_to_one 的 left 连接做质量审计。

    返回形状 / 行数守恒 / 匹配率 / 是否低于 0.95 阈值。
    """
    result = left.merge(right, on=key, how="left", validate="many_to_one")
    rate = round(float(result[probe_col].notna().mean()), 4)
    return {
        "形状": result.shape,
        "行数守恒": len(result) == len(left),
        "匹配率": rate,
        "低命中": rate < 0.95,
    }


print("dev 台区数:", dev[KEY].nunique(), "| info 台区数:", info[KEY].nunique())

---

## 三、逐节讲解

### 3.1 心智模型：连接是"按键对齐"，不是"按位置拼接"

```text
dev（主表，270 行，18 个台区，每台区多行）
  └─ 台区编号 ─┐
              ├─ merge on="台区编号" ─▶ 宽表
info（维表，16 行，每台区 1 行）
  └─ 台区编号 ─┘
```

**基数（cardinality）是连接的唯一核心概念**：

| 左表基数 | 右表基数 | 结果行数 | 说明 |
|---|---|---|---|
| 多 | 一 | **= 左表行数** | `many_to_one`，最常见（主表 + 维表） |
| 一 | 多 | **= 右表行数** | `one_to_many` |
| 一 | 一 | **= 键的个数** | `one_to_one` |
| 多 | 多 | **= 笛卡尔积** ⚠️ | `many_to_many`，行数会爆 |

本数据的基数是：`dev` 的 `台区编号` **多**（270 行 / 18 台区）、
`info` 的 `台区编号` **一**（16 行 / 16 台区）→ 所以 `how="left"` 的
结果行数应该**恰好等于 270**。这就是第一条校验公式。

In [ ]:
print(f"{'how':<8}{'结果形状':>14}{'行数是否 == 左表':>18}")
print("-" * 42)
for how in ["inner", "left", "right", "outer"]:
    result = dev.merge(info, on=KEY, how=how)
    print(f"{how:<8}{str(result.shape):>14}{str(len(result) == len(dev)):>18}")

print("\n左表台区数:", dev[KEY].nunique(), "| 维表台区数:", info[KEY].nunique())
missing = sorted(set(dev[KEY]) - set(info[KEY]))
print("主表有、维表没有的台区:", missing)
print("→ 这", len(missing), "个台区的", len(dev[dev[KEY].isin(missing)]), "行，在 inner 连接里被丢掉了")
print("   inner:", len(dev.merge(info, on=KEY, how="inner")),
      "| left:", len(dev.merge(info, on=KEY, how="left")),
      f"| 差值:", len(dev) - len(dev.merge(info, on=KEY, how="inner")))

**结论**：`inner` 悄悄丢了 **23** 行（`STATION_B_03` + `STATION_F_02` 的所有记录）。
如果要交一份"每台区统计表"，用 `inner` 会让这两个台区凭空消失——
而题目很可能要求"18 个台区都要有"。

> **判定规则**：**主表 + 维表一律用 `how="left"`**，
> 只在明确知道"业务上就该只保留双方都有的"时才用 `inner`。
> 而且用 `left` 之后要**主动检查 `NaN` 数**（见 §3.2）。

### 3.2 难点深挖①：键"看着一样"，却匹配 0 行

**为什么难**：这是连接里最贵的一个坑。`merge` 不报错、不警告、
结果表**列数不变、行数不变**，只有你新加的那一列**整列是 `NaN`**。
而 `NaN` 又很容易被归咎于"数据本身缺失"。

**错误示范**：把维表的键前面多加一个空格（模拟 Excel 里手打的隐形空白）

In [ ]:
dirty_key = info.copy()
dirty_key[KEY] = " " + dirty_key[KEY].astype(str)

clean_result = dev.merge(info, on=KEY, how="left")
dirty_result = dev.merge(dirty_key, on=KEY, how="left")

print(f"{'版本':<16}{'形状':>12}{'匹配行数':>10}{'匹配率':>10}")
print("-" * 48)
for name, result in [("干净键", clean_result), ("多一个空格", dirty_result)]:
    print(f"{name:<16}{str(result.shape):>12}"
          f"{int(result['所属供电所'].notna().sum()):>10}"
          f"{match_rate(result, '所属供电所'):>10.4f}")

print("\n→ 形状完全一样 (270, 17)，只有匹配率从 0.9148 掉到 0.0")
print("  如果不看匹配率，这个错误会被当成「维表数据缺失」放过")

print("\n三种典型脏键及修法:")
cases = {
    "首尾空白": lambda s: " " + s + " ",
    "全角空格": lambda s: "\u3000" + s,
    "大小写": lambda s: s.str.lower(),
}
for label, fn in cases.items():
    broken = info.copy()
    broken[KEY] = fn(broken[KEY])
    r = dev.merge(broken, on=KEY, how="left")
    print(f"  {label:<8} 匹配行数 = {int(r['所属供电所'].notna().sum()):<5}")

print("\n修法：连接前统一做一遍键规范化 —— "
      "`.astype(str).str.strip().str.replace('\u3000','').str.upper()`")

**正误对照**：

| 版本 | 形状 | 匹配行数 | 匹配率 |
|---|---|---|---|
| 干净键 | `(270, 17)` | **247** | **0.9148** |
| 键多一个空格 | `(270, 17)` | **0** | **0.0000** |
| 键有全角空格 | `(270, 17)` | 0 | 0.0 |
| 键被转小写 | `(270, 17)` | 0 | 0.0 |

**4 种情况形状全是 `(270, 17)`**——光看形状一个都发现不了。

> **判定规则（连接的铁律）**：
> **连接后立刻算一遍右表列的非空率**：
> ```python
> result["右表列"].notna().mean()      # 正常应该 ≈ 左表命中率
> ```
> 数字明显偏低 → 先查键的规范性（`strip` / 全角空格 / 大小写 / dtype），
> **不要**先去填缺失值。填完了照样对不上。
>
> 连接前的键规范化一行搞定：
> ```python
> for frame in (dev, info):
>     frame[KEY] = frame[KEY].astype(str).str.strip().str.replace("\u3000", "").str.upper()
> ```

### 3.3 难点深挖②：一对多被当成一对一 → 行数膨胀

**为什么难**：当**右表的键有重复**时，`merge` 会把左表每一行与
右表所有匹配行**两两配对**（笛卡尔积）。行数膨胀，但**不报错**。

一个小例子（3 行 × 3 行 → 5 行）：

```python
a = pd.DataFrame({"k": ["x", "x", "y"], "v": [1, 2, 3]})   # k=x 出现 2 次
b = pd.DataFrame({"k": ["x", "x", "y"], "w": [9, 8, 7]})   # k=x 出现 2 次
a.merge(b, on="k")            # → 5 行！k=x 变成 4 行（2 × 2）
```

**正确做法**：连接时用 `validate` **主动声明基数**，不合规让 pandas 直接报错。

In [ ]:
a = pd.DataFrame({"k": ["x", "x", "y"], "v": [1, 2, 3]})
b = pd.DataFrame({"k": ["x", "x", "y"], "w": [9, 8, 7]})

blow = a.merge(b, on="k")
print("a", a.shape, "× b", b.shape, "→ merge 结果", blow.shape)
print(f"  k == 'x' 的行数: {int((blow['k'] == 'x').sum())}（= 2 × 2 笛卡尔积）")
print(blow.to_string(index=False))

print("\n用 validate 主动声明基数:")
for v in ["one_to_one", "one_to_many", "many_to_one", "many_to_many"]:
    try:
        result = a.merge(b, on="k", validate=v)
        print(f"  validate={v:<14} OK   shape={result.shape}")
    except pd.errors.MergeError as exc:
        print(f"  validate={v:<14} MergeError: {str(exc).splitlines()[0][:64]}")

print("\n回到真实数据（dev: 多 / info: 一）:")
for v in ["many_to_one", "one_to_one"]:
    try:
        result = dev.merge(info, on=KEY, how="left", validate=v)
        print(f"  validate={v:<14} OK   shape={result.shape}")
    except pd.errors.MergeError as exc:
        print(f"  validate={v:<14} MergeError: {str(exc).splitlines()[0][:64]}")

> **判定规则**：**每次 `merge` 都带上 `validate`**。
> 它不改变结果，只是把"你没想清楚基数"这件事变成一个立刻可见的错误。
>
> - 主表 + 维表 → `validate="many_to_one"`（左多右一）
> - 两张明细表按键拼 → `validate="one_to_one"` 或 `"many_to_many"`
>
> 代价是零，收益是"行数膨胀"这类错误**再也不会静默发生**。
> 另一个便宜的护栏是 `indicator=True`，它会加一列 `_merge` 告诉你
> 每行是 `both` / `left_only` / `right_only`。

In [ ]:
flagged = dev.merge(info, on=KEY, how="left", indicator=True)
print("indicator 列的 dtype:", flagged["_merge"].dtype, "← pandas 3.0 是 category")
print("indicator 列的取值分布:")
print(flagged["_merge"].value_counts().to_string())
print("\n→", int((flagged["_merge"] == "left_only").sum()),
      "行是 left_only，正好等于 270 - 247 =", len(dev) - 247)
print("\n⚠️ 注意 value_counts 把 right_only = 0 也列出来了 ——")
print("   因为 _merge 是 category，value_counts 默认列出全部类别（含 0 计数）。")
print("   想跳过 0 计数：value_counts().pipe(lambda s: s[s > 0])")

print("\n用 indicator 直接定位「维表缺哪些台区」:")
print(sorted(flagged.loc[flagged["_merge"] == "left_only", KEY].unique()))

### 3.4 难点深挖③：`suffixes` 只管**非键**的同名列

**为什么难**：两表有同名列时，pandas 自动加 `_x` / `_y` 后缀
（或你指定的 `suffixes`），**但连接键不加**。所以你会看到
"有的同名列加了后缀、有的没加"——列名规则不一致，后面按名字取列就取错。

```python
a = dev[["记录ID", "台区编号", "负荷值"]].head(3)
b = dev[["记录ID", "台区编号", "负荷值"]].head(3)
a.merge(b, on="台区编号", suffixes=("_左", "_右"))
# 列名 → ['记录ID_左', '台区编号', '负荷值_左', '记录ID_右', '负荷值_右']
#                    ↑ 键没有被加后缀
```

**列序也反直觉**：左表的列（含键，键在它原来的位置）在前，
右表的非键列在后。

In [ ]:
left3 = dev[["记录ID", KEY, "负荷值"]].head(3)
right3 = dev[["记录ID", KEY, "负荷值"]].head(3)

print("左表列:", list(left3.columns))
print("右表列:", list(right3.columns))
print("\n默认 suffixes:")
print(" ", list(left3.merge(right3, on=KEY).columns))
print("自定义 suffixes=('_左','_右'):")
print(" ", list(left3.merge(right3, on=KEY, suffixes=("_左", "_右")).columns))
print("\n→ 键 '台区编号' 没有被加后缀；记录ID / 负荷值 都加了")
print("  所以「用列名取列」时绝不能想当然，先 print(columns) 再取")

> **判定规则**：两表有同名列时，**连接前先改名**，比事后靠后缀猜更安全：
> ```python
> # ✅ 推荐：连接前把右表列名改成带业务含义的名字
> info2 = info.rename(columns={"容量kVA": "容量"})
> # ❌ 事后猜：result["负荷值_x"] 到底是哪张表的？
> ```
> 中文列名做后缀也没问题（`suffixes=("_左", "_右")` 实测可用），
> 但**列名里的 `_x` / `_y` 一定要在交付前清掉**，否则交卷表格很难看。

### 3.5 主表 + 多张维表的链式拼装

真实任务几乎不会只拼一张表。本数据要拼成一份"带台账和台区汇总的宽表"：

```text
dev (270, 13)  +  info_cn (16, 5)  +  summary (18, 3)  →  (270, 19)
```

注意 **列数的加法**：`13 + 5 + 3 - 1 - 1 = 19`——
每次连接会**消耗掉一个重复的键列**。

按顺序一次拼完：

In [ ]:
wide = dev.merge(info_cn, on=KEY, how="left").merge(summary, on=KEY, how="left")

print("宽表形状:", wide.shape, "| 预期列数: 13 + 5 + 3 - 1 - 1 = 19")
print("\n新增的列:", [c for c in wide.columns if c not in dev.columns])
print("\n各新增列的 NaN 数:")
for col in ["所属供电所", "容量", "投运年份", "供电区域类型", "日均负荷", "户数"]:
    print(f"  {col:<10} NaN = {int(wide[col].isna().sum()):<4}"
          f"非空率 = {match_rate(wide, col):.4f}")

print("\n→ 台账 4 列都缺 23 行（B_03 / F_02 两个台区在维表里不存在）")
print("  summary 2 列一个都不缺（汇总表有全部 18 个台区）")
print("  这个「23 vs 0」的差异，就是这张宽表质量的全部真相")

**三条校验公式**（每次连接后都该跑一遍）：

| 公式 | 本数据 | 含义 |
|---|---|---|
| `len(result) == len(dev)` | **270 == 270** ✅ | `many_to_one` 连接行数守恒 |
| `result[KEY].nunique() == dev[KEY].nunique()` | **18 == 18** ✅ | 没有凭空多出台区 |
| `result[右表列].notna().mean()` | 台账 **0.9148** / 汇总 **1.0** | 命中率符合预期 |

如果第三条数字**远低于**你预期的命中率，别急着 `fillna`——先查键。

In [ ]:
checks = {
    "行数守恒": len(wide) == len(dev),
    "台区数守恒": wide[KEY].nunique() == dev[KEY].nunique(),
    "台账命中率符合预期": abs(match_rate(wide, "所属供电所") - 247 / 270) < 1e-9,
    "汇总命中率 == 1": match_rate(wide, "日均负荷") == 1.0,
    "原列一个不少": set(dev.columns).issubset(set(wide.columns)),
}
for name, ok in checks.items():
    print(f"  {'✅' if ok else '❌'} {name}")

print("\n列数对账: 13 + 5 + 3 - 1(重复键) - 1(重复键) =", 13 + 5 + 3 - 1 - 1,
      "| 实际:", wide.shape[1])

### 3.6 难点深挖④：`concat` 默认**保留原索引**

**为什么难**：`pd.concat([a, b])` 默认 `ignore_index=False`，
两段的原索引会**原样留着**，于是结果里出现重复索引。
重复索引的后果是 `loc` / `[]` 会**一次返回多行**，
而后续所有"按索引对齐"的操作都开始出错。

```python
c1 = dev.head(3)     # index [0, 1, 2]
c2 = dev.head(2)     # index [0, 1]
pd.concat([c1, c2])                      # index → [0, 1, 2, 0, 1]  ⚠️
pd.concat([c1, c2], ignore_index=True)   # index → [0, 1, 2, 3, 4]  ✅
```

In [ ]:
c1 = dev.head(3)
c2 = dev.head(2)

plain = pd.concat([c1, c2])
fixed = pd.concat([c1, c2], ignore_index=True)

print("c1 索引:", c1.index.tolist(), "| c2 索引:", c2.index.tolist())
print("concat 默认       索引:", plain.index.tolist(), "| 有重复:", plain.index.has_duplicates)
print("concat(ignore_index=True) 索引:", fixed.index.tolist(), "| 有重复:", fixed.index.has_duplicates)
print("长度都是:", len(plain), len(fixed))

print("\n重复索引的后果 —— 用 label 取值:")
print("  plain.loc[0]     取到", len(plain.loc[0]), "行（本想要 1 行）")
print("  用 iloc 才安全: plain.iloc[0] 取到 1 行")

print("\n横向 concat（axis=1）按【索引标签】对齐，不按位置:")
c1r = c1.reset_index(drop=True)
c2r = c2.reset_index(drop=True)
print("  c1r", c1r.shape, "| c2r", c2r.shape,
      "| concat(axis=1) →", pd.concat([c1r, c2r], axis=1).shape)
print("  少的那一段补齐 NaN，行数取并集/较长者")

**`concat` 的另外两个开关**：

| 参数 | 取值 | 作用 |
|---|---|---|
| `join` | `"outer"`（默认）/ `"inner"` | 列不同时取并集（补 `NaN`）还是交集 |
| `keys` | 列表 | 加一级索引区分来源（`MultiIndex`） |

列不一致时的默认行为是**并集 + 补 `NaN`**，这也是一个静默点：

In [ ]:
p1 = dev.head(3)[["记录ID", KEY]]
p2 = dev.head(3)[["记录ID", "负荷值"]]

outer = pd.concat([p1, p2])
inner = pd.concat([p1, p2], join="inner")

print("p1 列:", list(p1.columns), "| p2 列:", list(p2.columns))
print(f"  concat 默认(join='outer') → {outer.shape}  列: {list(outer.columns)}")
print(f"  concat(join='inner')      → {inner.shape}  列: {list(inner.columns)}")
print("\n默认结果的 NaN 数:")
print(outer.isna().sum().to_string())

keyed = pd.concat([c1, c2], keys=["第一批", "第二批"])
print("\nconcat(keys=[...]) 索引层级:", keyed.index.names)
print(keyed[["记录ID"]].head(4).to_string())

> **判定规则**：
> - 纵向堆叠**明细行**（多个文件、多天数据） → 一律 `ignore_index=True`
> - 想保留"数据来自哪一批" → 用 `keys=`，得到一级 `MultiIndex`，**不要**靠原索引猜
> - 横向拼列前，**先 `reset_index(drop=True)`**，否则是按标签对齐而不是按位置

---

### 3.7 难点深挖⑤：`merge_asof` —— 用"最近的一个"代替"等于"

**为什么难**：很多业务规则不是"键相等"，而是"**表里查不到就等于前一条生效的**"。
比如容量限值每半月调一次，但负荷是逐小时的：

```text
限值表：2026-01-01 → 1000 ┃ 2026-01-16 → 1050 ┃ 2026-02-01 → 1100 ...
负荷表：逐小时 2160 行，没有任何时间戳能"等于"限值表的生效时间
```

用 `merge` 会一行都匹配不上（时间戳不相等）。
必须用 `pd.merge_asof`——它找的是**不晚于（或不低于）该时刻的最近一条**。

**三个硬约束**：

1. 两边的键**必须已排序**，否则 `ValueError: left keys must be sorted`
2. `direction`：`"backward"`（默认，取 ≤ 的最近一条）/ `"forward"` / `"nearest"`
3. 支持 `by=` 分组（组内各自最近匹配），以及 `tolerance=` 容差

In [ ]:
asof = pd.merge_asof(
    hourly, limit, left_on="时间戳", right_on="生效时间", direction="backward"
)

print("merge_asof(backward) →", asof.shape, "| 限值 NaN:", int(asof["限值"].isna().sum()))
print("\n限值取值分布（按生效区间，每段应该是整数天 × 24 小时）:")
for value, count in asof["限值"].value_counts().sort_index().items():
    print(f"  限值 {value:<5} → {count:>4} 小时 = {count / 24:.0f} 天")

print("\n超限条数（负荷值 > 限值）:", int((asof["负荷值"] > asof["限值"]).sum()))

fwd = pd.merge_asof(
    hourly, limit, left_on="时间戳", right_on="生效时间", direction="forward"
)
print("\nmerge_asof(forward) → 限值 NaN:", int(fwd["限值"].isna().sum()),
      "（最后一个生效时间之后的都匹配不上）")
print("  forward 的分布:", {int(k): int(v) for k, v in fwd["限值"].value_counts().sort_index().items()})

print("\n未排序时（不报错才怪）:")
try:
    pd.merge_asof(hourly.sample(frac=1, random_state=0), limit,
                  left_on="时间戳", right_on="生效时间")
except ValueError as exc:
    print("  ->", type(exc).__name__, ":", exc)

**`backward` 与 `forward` 的语义差别**（这决定你有没有用未来数据）：

| `direction` | 取哪一条 | 本质 |
|---|---|---|
| `"backward"`（默认） | 生效时间 **≤** 当前时刻的最近一条 | **用了过去**——做特征安全 |
| `"forward"` | 生效时间 **≥** 当前时刻的最近一条 | **用了未来**——只有做"模拟当时可用信息"才合理 |
| `"nearest"` | 最近的一条（不分前后） | 可能取到未来，**默认别用** |

本数据实测：`backward` 的 `NaN` 是 **0**（全小时都能找到历史限值），
`forward` 的 `NaN` 是 **383**（最后一个生效时间之后的时段查不到"未来限值"）。

> **判定规则**：
> - 做**特征**、算"当时的规则是什么" → **`backward`**
> - 做**回测/复盘**"下一个决策点是什么" → `forward`，但要明确知道自己在用未来
> - `nearest` 在竞赛里几乎用不到，**误用会导致特征泄漏**

### 3.8 `join` 与 `merge` 的关系

`join` 是 `merge` 的**索引版快捷方式**：

```python
left.set_index(KEY).join(right.set_index(KEY), how="left")
# 完全等价于
left.merge(right, on=KEY, how="left")
```

什么时候用 `join`：右侧已经 `set_index` 过（比如做过多级索引分析），
不想再 `reset_index` 来回折腾。否则**一律用 `merge`**——
`merge` 的 `on=` / `validate=` / `indicator=` 都比 `join` 清楚。

In [ ]:
via_join = dev.set_index(KEY).join(info.set_index(KEY), how="left", rsuffix="_r")
via_merge = dev.merge(info, on=KEY, how="left")

print("join 结果:", via_join.shape, "| merge 结果:", via_merge.shape)
print("join 把键放到了索引里:", via_join.index.name, "→ 交付前记得 reset_index()")

# 值层面等价性对账
a = via_join["所属供电所"].reset_index(drop=True)
b = via_merge["所属供电所"]
common = a.notna() & b.notna()
print("\n两者非空位置上的值完全一致:", bool((a[common] == b[common]).all()))
print("非空数:", int(a.notna().sum()), "vs", int(b.notna().sum()))

---

### 3.9 综合：从"多张表"到"一张可交付宽表"

一份合格的交付宽表要满足：

| 检查项 | 判据 |
|---|---|
| 行数守恒 | `len(wide) == len(主表)` |
| 键不膨胀 | 台区数 == 主表台区数 |
| 命中率符合预期 | 每个新列的非空率都能解释 |
| 列名干净 | 没有 `_x` / `_y` / `_左` / `_右` 残留 |
| 索引干净 | `RangeIndex`，无重复 |
| 缺失有交代 | 每个 `NaN` 都能说出原因 |

下面一次性做完这六件事：

In [ ]:
deliver = (
    dev.merge(info_cn, on=KEY, how="left", validate="many_to_one")
    .merge(summary, on=KEY, how="left", validate="many_to_one")
)

# 列名清理：把可能残留的后缀去掉
deliver.columns = [c.replace("_x", "").replace("_y", "") for c in deliver.columns]
# 索引清理
deliver = deliver.reset_index(drop=True)

audit = {
    "行数": len(deliver),
    "台区数": int(deliver[KEY].nunique()),
    "列数": deliver.shape[1],
    "台账列缺行数": int(deliver["所属供电所"].isna().sum()),
    "汇总列缺行数": int(deliver["日均负荷"].isna().sum()),
    "索引无重复": not deliver.index.has_duplicates,
    "无后缀残留": not any(c.endswith(("_x", "_y", "_左", "_右")) for c in deliver.columns),
}
print("交付前自检:")
for name, value in audit.items():
    print(f"   {name:<14} = {value}")

print("\n缺失交代: 台账 4 列各缺", audit["台账列缺行数"],
      "行，来自", sorted(set(dev[KEY]) - set(info[KEY])), "两个台区")
print("→ 这两个台区确实不在台账里，属于「业务上就是没有」，保留 NaN 并写明即可")

**"缺失有交代"是评分的隐藏项**。三种缺失要在报告里区分开：

| 缺失来源 | 识别方法 | 处理 |
|---|---|---|
| 业务上不存在（本台区没台账） | 连接后 `NaN` + `indicator="left_only"` | **保留 `NaN`**，报告里说明 |
| 主表自身缺失（漏采） | 连接前就在 `NaN` | 按 ch04 的规则填充 |
| 键脏导致没匹配上 | 命中率远低于预期 | **先修键**，再重新连接 |

这三者在外表上完全一样（都是 `NaN`），但处理方式完全不同——
把"键脏"当成"业务缺失"填掉，是连接类错误最常见的收尾方式。

---

## 四、易错点清单

1. **主表 + 维表用了 `inner`** → 主表独有的键对应的行被静默丢掉（本例 **23** 行）
2. **键有隐形空白 / 全角空格 / 大小写差异** → 形状完全正常，但右表列**整列 `NaN`**
3. **不看右表列的非空率** → 上面这个错误会被当成"数据本身缺失"
4. **右表键有重复却按一对一用** → 行数按笛卡尔积膨胀，不报错
5. **不加 `validate`** → 放弃了"零成本"的基数护栏
6. **以为键也会加后缀** → `suffixes` 只作用于**非键**同名列，键保持原名
7. **`concat` 忘了 `ignore_index=True`** → 索引重复，`loc` 一次返回多行
8. **横向 `concat` 前没 `reset_index`** → 变成按标签对齐，错位却不出错
9. **`concat` 默认 `join="outer"`** → 列不同时静默多出 `NaN` 列
10. **`merge_asof` 前没排序** → `ValueError: left keys must be sorted`
11. **`merge_asof` 的 `direction` 选成 `forward`/`nearest`** → 用了未来数据
12. **连接后不清理 `_x` / `_y` 后缀** → 交付表格列名难看，且容易取错列
13. **用 `fillna(0)` 处理连接产生的 `NaN`** → 把"没台账"变成"台账值为 0"，语义被篡改
14. **`_merge` 是 `category`** → `value_counts()` 会列出 0 计数的类别，别误判成"真的有这类行"

---

## 五、本章小结

| 我想…… | 用哪个 | 关键参数 |
|---|---|---|
| 主表挂维表字段 | `dev.merge(info, on=KEY, how="left")` | `validate="many_to_one"` |
| 两表键名不同 | `merge(left_on=, right_on=)` | — |
| 用索引连接 | `left.join(right)` 或 `left_index=` | — |
| 追加多批明细行 | `pd.concat([...], ignore_index=True)` | `keys=` 标记批次 |
| 横向加列 | `pd.concat([...], axis=1)` | **先 `reset_index(drop=True)`** |
| 查"当时生效的规则" | `pd.merge_asof(..., direction="backward")` | 两边都要排序 |
| 反查连接是否命中 | `result[右表列].notna().mean()` | 或 `indicator=True` |

**一句话总结**：连接的所有错误都指向同一件事——
**你没有验证"这次连接的基数对不对、命中率对不对"**。
所以每次 `merge` 之后固定跑三条：`行数守恒` / `键不膨胀` / `右表列非空率`。
三行代码，能挡掉本章 90% 的坑。